# PathSnap – 20 features + 8 models + HMM/Viterbi

**Run in Google Colab**
1. Upload your data zip (e.g. `Train_Test.zip`, with `train` and `test` trip folders) to `/content` (Files sidebar). If you skip this, the data step opens an upload dialog.
2. Runtime → Run all. The training cell prints progress per fold and per model, with elapsed time.

**Quick test:** in the config cell set `QUICK_MODE = True` (fewer points/folds, smaller models). Leave `False` for report numbers.

**Models:** Logistic Regression, Gaussian NB, Random Forest, Extra Trees, LightGBM, XGBoost, MLP, calibrated Linear SVM.  
**HMM:** hidden state = candidate arc; emission = log model probability; transition = topology penalty (same / n hops / U-turn / disconnected); chain reset on GPS gaps; decoder tuned on out-of-fold train predictions only.

In [11]:
import os, sys, json, time, zipfile, warnings, subprocess, itertools
from pathlib import Path

import numpy as np
import pandas as pd


## Config

In [12]:
DATA_DIR = None              # e.g. "/content/drive/MyDrive/Train_Test"; None = auto (zip upload)
WORK_DIR = Path("/content/pathsnap_work") if "google.colab" in sys.modules else Path("./pathsnap_work")
OUTPUT_DIR = WORK_DIR / "outputs"
SEED = 42
QUICK_MODE = False   # set True for a fast smoke test (less data, small models); False = full run

CAND_RADIUS_M = 60.0         # candidate search radius around each GPS point
MAX_CANDIDATES = 15          # keep the K nearest arcs per point
ALIGN_RADIUS_M = 60.0        # radius for aligning points to the reference route
LABEL_MAX_DIST_M = 50.0      # points further than this from their aligned route arc = unreliable label
MAX_POINTS_PER_TRIP = 120 if QUICK_MODE else 400    # evenly sampled points per trip (classifier training rows)
CV_FOLDS = 3 if QUICK_MODE else 5
SEQ_LEN_TRAIN = 200 if QUICK_MODE else 600          # consecutive points per train trip used for HMM tuning
SEQ_LEN_TEST = 600 if QUICK_MODE else None          # None = decode the whole test trips
MAX_HOPS = 3                 # topology: arcs reachable within this many hops count as "connected"
GAP_S = 30.0                 # time gap above which the HMM chain is reset
USE_CACHE = True

DECODER_GRID = dict(                                    # tuned on out-of-fold TRAIN predictions only
    hop_pen=[1.0] if QUICK_MODE else [0.5, 1.5],
    jump_pen=[4.0, 10.0] if QUICK_MODE else [3.0, 6.0, 10.0],
    uturn_pen=[3.0] if QUICK_MODE else [3.0, 8.0],
    emission_weight=[1.0],
)

warnings.filterwarnings("ignore")


def _ensure(pkg, import_name=None, min_version=None):
    try:
        mod = __import__(import_name or pkg)
        if min_version and tuple(int(x) for x in mod.__version__.split(".")[:2]) < min_version:
            raise ImportError
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pkg])


_ensure("shapely", min_version=(2, 0))
_ensure("lightgbm")
_ensure("xgboost")

import shapely                                   # noqa: E402
from shapely.strtree import STRtree              # noqa: E402
from sklearn.model_selection import GroupKFold   # noqa: E402
from sklearn.pipeline import make_pipeline       # noqa: E402
from sklearn.preprocessing import StandardScaler # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.naive_bayes import GaussianNB       # noqa: E402
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier  # noqa: E402
from sklearn.neural_network import MLPClassifier # noqa: E402
from sklearn.svm import LinearSVC                # noqa: E402
from sklearn.calibration import CalibratedClassifierCV  # noqa: E402
from sklearn.metrics import (log_loss, accuracy_score, precision_recall_fscore_support,  # noqa: E402
                             f1_score, confusion_matrix)
from lightgbm import LGBMClassifier              # noqa: E402
from xgboost import XGBClassifier                # noqa: E402

FEATURES = [
    # distance to the candidate
    "dist", "dist_rank", "dist_minus_min", "n_cands", "signed_offset",
    # direction
    "head_cos", "head_valid", "turn_cos",
    # position along the arc / arc properties
    "frac", "arc_len", "has_reverse",
    # motion
    "speed", "dt_prev", "along_move",
    # neighbouring GPS points vs the same candidate
    "dist_prev", "dist_next", "dist_prev3", "dist_next3", "dist_nb_mean",
    # topology
    "connect_prev",
]
assert len(FEATURES) == 20

HOP_NAMES = ["same", "1hop", "2hop", "3hop", "uturn", "disconnected"]   # transition codes 0..5


def fmt(sec):
    sec = int(sec)
    return f"{sec // 60}m{sec % 60:02d}s" if sec >= 60 else f"{sec}s"


## Data Location

In [13]:
def _extract_zip(zip_path: Path) -> Path:
    out = WORK_DIR / "data"
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            if m.startswith("__MACOSX") or os.path.basename(m).startswith("._"):
                continue
            z.extract(m, out)
    return out


def resolve_data_root() -> Path:
    if DATA_DIR and Path(DATA_DIR).exists():
        return Path(DATA_DIR)
    for p in ["/content/Train_Test", "/content/Train&Test", "/content/drive/MyDrive/Train_Test",
              "/content/drive/MyDrive/Train&Test", "./Train_Test", "./Train&Test"]:
        if Path(p).is_dir():
            return Path(p)
    zips = list(Path("/content").glob("*.zip")) + list(Path(".").glob("*.zip"))
    zips = [z for z in zips if "train" in z.name.lower() or "test" in z.name.lower()] or zips
    if not zips and "google.colab" in sys.modules:
        from google.colab import files
        print("Upload your Train_Test.zip ...")
        up = files.upload()
        zips = [Path(n) for n in up if n.lower().endswith(".zip")]
    if not zips:
        raise FileNotFoundError("No data found. Put Train_Test.zip next to this script, or set DATA_DIR.")
    print(f"Extracting {zips[0]} ...")
    return _extract_zip(zips[0])


def find_trips(root: Path):
    trips = []
    for p in sorted(root.rglob("*.track")):
        rel = p.relative_to(root).parts
        if "__MACOSX" in rel or p.name.startswith("._"):
            continue
        low = [x.lower() for x in rel[:-1]]
        split = "train" if "train" in low else "test" if "test" in low else None
        if split is None:
            print(f"  skipping (not in a train/test folder): {p}")
            continue
        trips.append((split, p.stem, p))
    return trips


## Road-Network Topology Helpers

In [14]:
class ArcGraph:
    """Directed arc graph: topological relation between two arcs (used by a feature and by the HMM)."""

    def __init__(self, A, B, n_nodes):
        self.A, self.B = A, B
        self.order = np.argsort(A, kind="stable")
        self.ptr = np.searchsorted(A[self.order], np.arange(n_nodes + 1))
        self.memo = {}

    def _out(self, node):
        return self.order[self.ptr[node]:self.ptr[node + 1]]

    def _reach(self, a):
        r = self.memo.get(a)
        if r is None:
            r, frontier = {}, [a]
            for h in range(1, MAX_HOPS + 1):
                nxt = []
                for x in frontier:
                    for y in self._out(self.B[x]):
                        y = int(y)
                        if y != a and y not in r:
                            r[y] = h
                            nxt.append(y)
                frontier = nxt
            self.memo[a] = r
        return r

    def code(self, a, b):
        """0 same arc | 1..MAX_HOPS forward hops | 4 U-turn (reverse arc) | 5 not connected."""
        a, b = int(a), int(b)
        if a == b:
            return 0
        if self.A[b] == self.B[a] and self.B[b] == self.A[a]:
            return 4
        h = self._reach(a).get(b)
        return h if h is not None else 5

    def matrix(self, prev_arcs, cur_arcs):
        return np.array([[self.code(a, b) for b in cur_arcs] for a in prev_arcs], dtype=np.int8)


## Per-Trip Processing

In [15]:
def project(lon, lat, lon0, lat0):
    """Local equirectangular projection to metres (accurate for city-scale trips; per-trip origin)."""
    return np.c_[(lon - lon0) * np.cos(np.radians(lat0)) * 111320.0, (lat - lat0) * 110540.0]


def align_to_route(pt_geoms, route_lines):
    """Monotone alignment of each GPS point to an arc of the reference route (Viterbi-style DP).
    Returns the index into the route for each point. Route order is respected (no going backwards)."""
    n = len(pt_geoms)
    tree = STRtree(route_lines)
    qi, ti = tree.query(pt_geoms, predicate="dwithin", distance=ALIGN_RADIUS_M)
    d = shapely.distance(pt_geoms[qi], route_lines[ti])
    order = np.argsort(qi, kind="stable")
    qi, ti, d = qi[order], ti[order], d[order]
    starts = np.searchsorted(qi, np.arange(n + 1))
    cj, cd = [None] * n, [None] * n
    for i in range(n):
        s, e = starts[i], starts[i + 1]
        if e > s:
            cj[i], cd[i] = ti[s:e], d[s:e]
    miss = np.where(starts[1:] == starts[:-1])[0]
    if len(miss):
        near = tree.nearest(pt_geoms[miss])
        dm = shapely.distance(pt_geoms[miss], route_lines[near])
        for k, i in enumerate(miss):
            cj[i], cd[i] = np.array([near[k]]), np.array([dm[k]])
    back = []
    prev_j, prev_cost = cj[0], cd[0].astype(float)
    for i in range(1, n):
        j, dd = cj[i], cd[i]
        diff = j[:, None] - prev_j[None, :]
        pen = np.where(diff >= 0, 2.0 * np.maximum(diff - 1, 0), 30.0 * (-diff))
        tot = prev_cost[None, :] + pen
        bi = tot.argmin(1)
        prev_cost = dd + tot[np.arange(len(j)), bi]
        back.append(bi)
        prev_j = j
    path = np.zeros(n, dtype=np.int64)
    k = int(prev_cost.argmin())
    path[n - 1] = cj[n - 1][k]
    for i in range(n - 1, 0, -1):
        k = int(back[i - 1][k])
        path[i - 1] = cj[i - 1][k]
    return path


def process_trip(split, tid, track_path: Path):
    base = track_path.with_suffix("")
    tr = np.loadtxt(track_path, ndmin=2)
    nodes = np.loadtxt(base.with_suffix(".nodes"), ndmin=2)
    arcs = np.loadtxt(base.with_suffix(".arcs"), dtype=np.int64, ndmin=2)
    route = np.loadtxt(base.with_suffix(".route"), dtype=np.int64, ndmin=1)

    tr = tr[np.isfinite(tr).all(1)]
    if not (np.diff(tr[:, 2]) >= 0).all():
        tr = tr[np.argsort(tr[:, 2], kind="stable")]
    assert arcs.max() < len(nodes) and route.max() < len(arcs), "index out of range"

    lon0, lat0 = np.median(tr[:, 0]), np.median(tr[:, 1])
    pts = project(tr[:, 0], tr[:, 1], lon0, lat0)
    nxy = project(nodes[:, 0], nodes[:, 1], lon0, lat0)
    t = tr[:, 2]
    n = len(pts)

    A, B = arcs[:, 0], arcs[:, 1]
    lines = shapely.linestrings(np.stack([nxy[A], nxy[B]], axis=1))
    arc_len = shapely.length(lines)
    arc_bearing = np.arctan2(nxy[B, 1] - nxy[A, 1], nxy[B, 0] - nxy[A, 0])
    key = A * len(nodes) + B
    has_rev = np.isin(B * len(nodes) + A, key).astype(np.float32)
    pt_geoms = shapely.points(pts)
    G = ArcGraph(A, B, len(nodes))
    tree = STRtree(lines)

    # ---- reference label per point (assumption: nearest route arc, in route order)
    j = align_to_route(pt_geoms, lines[route])
    ref_arc = route[j]
    ref_dist = shapely.distance(pt_geoms, lines[ref_arc])
    reliable = ref_dist <= LABEL_MAX_DIST_M

    # ---- per-point kinematics
    lo, hi = np.clip(np.arange(n) - 2, 0, n - 1), np.clip(np.arange(n) + 2, 0, n - 1)
    vec, dtw = pts[hi] - pts[lo], t[hi] - t[lo]
    mv = np.hypot(vec[:, 0], vec[:, 1])
    ok = (dtw > 0) & (dtw <= 20)
    speed = np.where(ok, mv / np.where(dtw > 0, dtw, 1), 0.0)
    head_valid = ok & (mv > 3.0)
    heading = np.arctan2(vec[:, 1], vec[:, 0])
    dt_prev = np.r_[0.0, np.diff(t)]
    i0, i1 = np.clip(np.arange(n) - 3, 0, n - 1), np.clip(np.arange(n) + 3, 0, n - 1)
    vin, vout = pts - pts[i0], pts[i1] - pts
    nin, nout = np.hypot(vin[:, 0], vin[:, 1]), np.hypot(vout[:, 0], vout[:, 1])
    cosv = (vin * vout).sum(1) / np.maximum(nin * nout, 1e-9)
    turn_cos = np.where((nin > 2) & (nout > 2), cosv, 1.0)       # 1 = straight, <1 = turning

    def make_cands(idx):
        """Candidate rows + 20 features for the GPS points `idx` (sorted by point, then distance)."""
        qi, ti = tree.query(pt_geoms[idx], predicate="dwithin", distance=CAND_RADIUS_M)
        if len(qi) == 0:
            return None
        pi = idx[qi]
        c = pd.DataFrame({"pt_idx": pi, "arc_id": ti, "dist": shapely.distance(pt_geoms[pi], lines[ti])})
        c = c.sort_values(["pt_idx", "dist"], kind="stable")
        c["dist_rank"] = c.groupby("pt_idx").cumcount()
        c = c[c.dist_rank < MAX_CANDIDATES].reset_index(drop=True)
        c["n_cands"] = c.groupby("pt_idx")["dist"].transform("size")
        c["dist_minus_min"] = c["dist"] - c.groupby("pt_idx")["dist"].transform("min")
        p, a = c["pt_idx"].values, c["arc_id"].values
        c["head_cos"] = np.where(head_valid[p], np.cos(heading[p] - arc_bearing[a]), 0.0)
        c["head_valid"] = head_valid[p].astype(np.float32)
        c["turn_cos"] = turn_cos[p]
        c["frac"] = shapely.line_locate_point(lines[a], pt_geoms[p], normalized=True)
        c["arc_len"] = arc_len[a]
        c["speed"] = speed[p]
        c["dt_prev"] = dt_prev[p]
        c["has_reverse"] = has_rev[a]
        ax_, ay_ = nxy[A[a], 0], nxy[A[a], 1]
        cross = (nxy[B[a], 0] - ax_) * (pts[p, 1] - ay_) - (nxy[B[a], 1] - ay_) * (pts[p, 0] - ax_)
        c["signed_offset"] = np.sign(cross) * c["dist"]
        for name, off in [("prev", -1), ("next", 1), ("prev3", -3), ("next3", 3)]:
            q = np.clip(p + off, 0, n - 1)
            c[f"dist_{name}"] = shapely.distance(pt_geoms[q], lines[a])
            if abs(off) == 1:
                c[f"frac_{name}"] = shapely.line_locate_point(lines[a], pt_geoms[q], normalized=True)
        c["dist_nb_mean"] = c[["dist_prev", "dist_next", "dist_prev3", "dist_next3"]].mean(1)
        c["along_move"] = (c["frac_next"] - c["frac_prev"]) * c["arc_len"]
        up = np.unique(p)                                             # topology vs previous point's nearest arc
        near_prev = tree.nearest(pt_geoms[np.clip(up - 1, 0, n - 1)])[np.searchsorted(up, p)]
        c["connect_prev"] = np.minimum([G.code(x, y) for x, y in zip(near_prev, a)], 4).astype(np.float32)
        c["label"] = (a == ref_arc[p]).astype(np.int8)
        return c[["pt_idx", "arc_id", "label"] + FEATURES]

    def tag(c):
        c.insert(0, "trip_id", tid)
        c.insert(1, "split", split)
        return c

    # ---- (a) classifier training/eval rows: evenly sampled reliable points
    rel_idx = np.where(reliable)[0]
    if len(rel_idx) > MAX_POINTS_PER_TRIP:
        rel_idx = rel_idx[np.linspace(0, len(rel_idx) - 1, MAX_POINTS_PER_TRIP).astype(int)]
    c = make_cands(rel_idx)
    if c is None:
        raise ValueError("no candidates generated")
    c = tag(c)
    hit = set(c.loc[c.label == 1, "pt_idx"].tolist())
    obs = pd.DataFrame({"trip_id": tid, "split": split, "pt_idx": rel_idx,
                        "hit": [int(i) in hit for i in rel_idx]})

    # ---- (b) HMM sequence: consecutive points (a window for train trips, whole trip for test)
    seq_len = SEQ_LEN_TRAIN if split == "train" else SEQ_LEN_TEST
    if seq_len is None or seq_len >= n:
        s0, s1 = 0, n
    else:
        s0 = int(0.3 * (n - seq_len))
        s1 = s0 + seq_len
    win = np.arange(s0, s1)
    cs = make_cands(win)
    meta, seq_df = None, None
    if cs is not None:
        cs = tag(cs)
        pt = cs.pt_idx.values
        uniq, first = np.unique(pt, return_index=True)
        starts = np.r_[first, len(pt)]
        arcs_all = cs.arc_id.values
        cand_arcs = [arcs_all[starts[k]:starts[k + 1]] for k in range(len(uniq))]
        Hm, gap = [None] * len(uniq), np.zeros(len(uniq), bool)
        for k in range(1, len(uniq)):
            if t[uniq[k]] - t[uniq[k - 1]] > GAP_S:
                gap[k] = True
            else:
                Hm[k] = G.matrix(cand_arcs[k - 1], cand_arcs[k])
        meta = dict(trip_id=tid, split=split, steps=uniq, cand_arcs=cand_arcs, H=Hm, gap=gap,
                    step_start=starts, ref_arc=ref_arc[uniq], reliable=reliable[uniq],
                    n_rel_window=int(reliable[win].sum()), n_rows=len(cs))
        seq_df = cs

    info = dict(trip_id=tid, split=split, points=n, nodes=len(nodes), arcs=len(arcs), route_arcs=len(route),
                unreliable_labels=int((~reliable).sum()), sampled_points=len(rel_idx),
                max_time_gap_s=float(np.diff(t).max()) if n > 1 else 0.0,
                pct_1s_steps=float((np.diff(t) == 1).mean()) if n > 1 else 1.0)
    return c, obs, info, seq_df, meta


def build_dataset():
    cache = OUTPUT_DIR / (f"dataset_cache_v2_{MAX_POINTS_PER_TRIP}_{int(CAND_RADIUS_M)}_{MAX_CANDIDATES}_"
                          f"{SEQ_LEN_TRAIN}_{SEQ_LEN_TEST}.pkl")
    if USE_CACHE and cache.exists():
        print(f"Loading cached dataset: {cache}")
        return pd.read_pickle(cache)
    root = resolve_data_root()
    trips = find_trips(root)
    print(f"Found {sum(s == 'train' for s, _, _ in trips)} train trips and "
          f"{sum(s == 'test' for s, _, _ in trips)} test trips under {root}")
    if not trips:
        raise FileNotFoundError("No .track files found - check the folder structure (train/ and test/).")
    C, O, I, S, M = [], [], [], [], []
    t0 = time.time()
    for k, (split, tid, path) in enumerate(trips, 1):
        try:
            c, o, i, s, m = process_trip(split, tid, path)
            C.append(c); O.append(o); I.append(i)
            if m is not None:
                S.append(s); M.append(m)
        except Exception as e:
            print(f"  ! skipped {split}/{tid}: {type(e).__name__}: {e}")
        if k % 10 == 0 or k == len(trips):
            el = time.time() - t0
            print(f"  processed {k}/{len(trips)} trips | elapsed {fmt(el)} | est. remaining {fmt(el / k * (len(trips) - k))}")
    off = 0
    for m in M:
        m["row_offset"] = off
        off += m["n_rows"]
    data = (pd.concat(C, ignore_index=True), pd.concat(O, ignore_index=True), pd.DataFrame(I),
            pd.concat(S, ignore_index=True), M)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(data, cache)
    return data


## Models

In [16]:
def get_models():
    q = QUICK_MODE
    cfg = {
        "1_LogisticRegression": ("Linear/probabilistic", dict(C=1.0, max_iter=500)),
        "2_GaussianNB": ("Linear/probabilistic", dict(var_smoothing=1e-9)),
        "3_RandomForest": ("Ensemble", dict(n_estimators=60 if q else 200, max_depth=16, min_samples_leaf=5)),
        "4_ExtraTrees": ("Tree/Ensemble", dict(n_estimators=60 if q else 200, max_depth=18, min_samples_leaf=5)),
        "5_LightGBM": ("Boosting", dict(n_estimators=150 if q else 400, learning_rate=0.05, num_leaves=31,
                                        subsample=0.8, subsample_freq=1, colsample_bytree=0.8)),
        "6_XGBoost": ("Boosting", dict(n_estimators=150 if q else 400, learning_rate=0.08, max_depth=6,
                                       subsample=0.8, colsample_bytree=0.8, tree_method="hist")),
        "7_MLP": ("Neural network", dict(hidden_layer_sizes=(64, 32), alpha=1e-4, max_iter=200,
                                         early_stopping=True)),
        "8_LinearSVM_calibrated": ("Kernel-based (linear)", dict(C=0.5, cv=3)),
    }
    factories = {
        "1_LogisticRegression": lambda p: make_pipeline(StandardScaler(), LogisticRegression(random_state=SEED, **p)),
        "2_GaussianNB": lambda p: make_pipeline(StandardScaler(), GaussianNB(**p)),
        "3_RandomForest": lambda p: RandomForestClassifier(random_state=SEED, n_jobs=-1, **p),
        "4_ExtraTrees": lambda p: ExtraTreesClassifier(random_state=SEED, n_jobs=-1, **p),
        "5_LightGBM": lambda p: LGBMClassifier(random_state=SEED, verbose=-1, **p),
        "6_XGBoost": lambda p: XGBClassifier(random_state=SEED, n_jobs=-1, verbosity=0, **p),
        "7_MLP": lambda p: make_pipeline(StandardScaler(), MLPClassifier(random_state=SEED, **p)),
        "8_LinearSVM_calibrated": lambda p: make_pipeline(
            StandardScaler(),
            CalibratedClassifierCV(LinearSVC(C=p["C"], dual=False, random_state=SEED), cv=p["cv"], method="sigmoid")),
    }
    return {k: (cfg[k][0], cfg[k][1], factories[k]) for k in cfg}


## Classifier Evaluation

In [17]:
def evaluate(cands: pd.DataFrame, obs: pd.DataFrame, proba: np.ndarray):
    y = cands["label"].values
    pred = (proba >= 0.5).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    tmp = pd.DataFrame({"trip_id": cands["trip_id"].values, "pt_idx": cands["pt_idx"].values,
                        "p": proba, "y": y})
    best = tmp.loc[tmp.groupby(["trip_id", "pt_idx"])["p"].idxmax()]
    correct = int(best["y"].sum())
    n_obs, n_hit = len(obs), int(obs["hit"].sum())
    return dict(
        accuracy=accuracy_score(y, pred), precision=pr, recall=rc, f1=f1,
        macro_f1=f1_score(y, pred, average="macro"),
        log_loss=log_loss(y, np.clip(proba, 1e-6, 1 - 1e-6), labels=[0, 1]),
        top1_acc=correct / max(n_obs, 1),                      # misses count as wrong (primary)
        top1_acc_given_hit=correct / max(n_hit, 1),
        candidate_miss_rate=1 - n_hit / max(n_obs, 1),
        n_obs=n_obs, n_candidate_rows=len(cands)), confusion_matrix(y, pred, labels=[0, 1])


## Hmm / Viterbi Route Reconstruction

In [18]:
def transition_table(hop_pen, jump_pen, uturn_pen):
    """log-transition score for each topology code: same, 1/2/3 hops, U-turn, disconnected (never -inf,
    so an invalid transition is expensive but the decoder can always continue)."""
    return np.array([0.0, -hop_pen, -2 * hop_pen, -3 * hop_pen, -uturn_pen, -jump_pen])


def viterbi_trip(meta, proba, table, w):
    """Viterbi over the candidate arcs of one trip. Returns (hmm choice idx, classifier-only idx) per step."""
    st = meta["step_start"]
    n = len(meta["cand_arcs"])
    em = [w * np.log(np.clip(proba[st[i]:st[i + 1]], 1e-6, 1.0)) for i in range(n)]
    clf = np.array([int(e.argmax()) for e in em])
    delta, back = em[0], [None] * n
    for i in range(1, n):
        H = meta["H"][i]
        if H is None:                                           # GPS gap / chain reset
            back[i] = np.full(len(em[i]), int(delta.argmax()))
            delta = delta.max() + em[i]
        else:
            sc = delta[:, None] + table[H]
            bi = sc.argmax(0)
            delta = sc[bi, np.arange(len(bi))] + em[i]
            back[i] = bi
    path = np.zeros(n, dtype=np.int64)
    k = int(delta.argmax())
    path[n - 1] = k
    for i in range(n - 1, 0, -1):
        k = int(back[i][k])
        path[i - 1] = k
    return path, clf


def seq_metrics(metas, proba_all, table, w, which):
    """Pointwise Top-1 (reliable points; no-candidate points count as wrong), continuity, route-arc F1."""
    correct = total = pairs = connected = 0
    f1s = []
    for m in metas:
        pr = proba_all[m["row_offset"]:m["row_offset"] + m["n_rows"]]
        hmm_idx, clf_idx = viterbi_trip(m, pr, table, w)
        idx = hmm_idx if which == "hmm" else clf_idx
        arcs = np.array([m["cand_arcs"][i][k] for i, k in enumerate(idx)])
        correct += int(((arcs == m["ref_arc"]) & m["reliable"]).sum())
        total += m["n_rel_window"]
        for i in range(1, len(idx)):
            if m["H"][i] is not None:
                pairs += 1
                connected += int(m["H"][i][idx[i - 1], idx[i]] <= MAX_HOPS)
        P, R = set(arcs.tolist()), set(m["ref_arc"][m["reliable"]].tolist())
        if P and R:
            pp, rr = len(P & R) / len(P), len(P & R) / len(R)
            f1s.append(0 if pp + rr == 0 else 2 * pp * rr / (pp + rr))
    return dict(top1=correct / max(total, 1), continuity=connected / max(pairs, 1),
                route_f1=float(np.mean(f1s)) if f1s else 0.0)


def tune_decoder(metas, proba_oof):
    rows = []
    for hp, jp, up, w in itertools.product(*DECODER_GRID.values()):
        tab = transition_table(hp, jp, up)
        r = seq_metrics(metas, proba_oof, tab, w, "hmm")
        rows.append(dict(hop_pen=hp, jump_pen=jp, uturn_pen=up, emission_weight=w, **r))
    g = pd.DataFrame(rows).sort_values(["top1", "continuity"], ascending=False).reset_index(drop=True)
    return g


## Main

In [19]:
def main():
    np.random.seed(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    T0 = time.time()
    cands, obs, info, seq_df, metas = build_dataset()

    print("\n=== DATASET INVENTORY ===")
    inv = info.groupby("split").agg(trips=("trip_id", "count"), gps_points=("points", "sum"),
                                    median_points=("points", "median"), median_nodes=("nodes", "median"),
                                    median_arcs=("arcs", "median"), unreliable_labels=("unreliable_labels", "sum"),
                                    max_gap_s=("max_time_gap_s", "max"), mean_pct_1s=("pct_1s_steps", "mean"))
    print(inv.round(2).to_string())
    for s in ["train", "test"]:
        o = obs[obs.split == s]
        print(f"[{s}] sampled observations: {len(o):,} | candidate rows: {(cands.split == s).sum():,} | "
              f"candidate miss rate: {1 - o.hit.mean():.3%}")
    print(f"Features used: {len(FEATURES)}  ->  {FEATURES}")
    info.to_csv(OUTPUT_DIR / "trip_inventory.csv", index=False)
    inv.to_csv(OUTPUT_DIR / "dataset_inventory.csv")

    tr_c, te_c = cands[cands.split == "train"].reset_index(drop=True), cands[cands.split == "test"].reset_index(drop=True)
    tr_o, te_o = obs[obs.split == "train"], obs[obs.split == "test"]
    Xtr, ytr, Xte = tr_c[FEATURES].values.astype(np.float32), tr_c["label"].values, te_c[FEATURES].values.astype(np.float32)
    groups = tr_c["trip_id"].values

    seq_tr_mask = (seq_df.split == "train").values
    seq_te_mask = ~seq_tr_mask
    Xs_tr, Xs_te = seq_df.loc[seq_tr_mask, FEATURES].values.astype(np.float32), seq_df.loc[seq_te_mask, FEATURES].values.astype(np.float32)
    seq_tr_trip = seq_df.loc[seq_tr_mask, "trip_id"].values
    # metas re-based so that row_offset is relative to the train-only / test-only seq tables
    def rebase(ms):
        out, off = [], 0
        for m in ms:
            m2 = dict(m); m2["row_offset"] = off; off += m["n_rows"]; out.append(m2)
        return out
    metas_tr = rebase([m for m in metas if m["split"] == "train"])
    metas_te = rebase([m for m in metas if m["split"] == "test"])

    try:
        splitter = GroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    except TypeError:
        splitter = GroupKFold(n_splits=CV_FOLDS)
    folds = list(splitter.split(Xtr, ytr, groups))

    models = get_models()
    fold_rows, summary, test_conf = [], [], {}
    oof_seq, test_seq = {}, {}
    print(f"\n=== TRAINING 8 MODELS ({CV_FOLDS}-fold group CV on {len(set(groups))} train trips, "
          f"then test on {te_o.trip_id.nunique()} student-test trips) ===")
    t_models = time.time()
    for mi, (name, (family, params, make)) in enumerate(models.items()):
        t_start = time.time()
        fold_metrics = []
        oof = np.zeros(len(Xs_tr))
        for f, (tri, vai) in enumerate(folds):
            m = make(params)
            m.fit(Xtr[tri], ytr[tri])
            vc = tr_c.iloc[vai]
            val_trips = set(vc.trip_id)
            vo = tr_o[tr_o.trip_id.isin(val_trips)]
            met, _ = evaluate(vc, vo, m.predict_proba(Xtr[vai])[:, 1])
            sm = np.isin(seq_tr_trip, list(val_trips))
            if sm.any():
                oof[sm] = m.predict_proba(Xs_tr[sm])[:, 1]
            fold_metrics.append(met)
            fold_rows.append(dict(model=name, fold=f, **met))
            print(f"   {name} fold {f + 1}/{CV_FOLDS} done | model elapsed {fmt(time.time() - t_start)}", flush=True)
        oof_seq[name] = oof
        cv = pd.DataFrame(fold_metrics)
        t1 = time.time()
        final = make(params).fit(Xtr, ytr)
        train_time = time.time() - t1
        t2 = time.time()
        proba = final.predict_proba(Xte)[:, 1]
        infer_time = time.time() - t2
        test_seq[name] = final.predict_proba(Xs_te)[:, 1]
        tmet, cm = evaluate(te_c, te_o, proba)
        test_conf[name] = cm.tolist()
        row = dict(model=name, family=family, config=json.dumps(params),
                   cv_top1_mean=cv.top1_acc.mean(), cv_top1_std=cv.top1_acc.std(),
                   cv_f1_mean=cv.f1.mean(), cv_logloss_mean=cv.log_loss.mean(), cv_logloss_std=cv.log_loss.std(),
                   cv_miss_rate=cv.candidate_miss_rate.mean(),
                   **{f"test_{k}": v for k, v in tmet.items() if k not in ("n_obs", "n_candidate_rows")},
                   train_time_s=train_time, infer_time_s=infer_time, infer_us_per_row=1e6 * infer_time / len(Xte),
                   total_cv_time_s=time.time() - t_start)
        summary.append(row)
        done = mi + 1
        el = time.time() - t_models
        print(f"{name:26s} CV top1 {row['cv_top1_mean']:.4f}±{row['cv_top1_std']:.4f} | "
              f"CV logloss {row['cv_logloss_mean']:.4f} | TEST top1 {row['test_top1_acc']:.4f} "
              f"F1 {row['test_f1']:.4f} logloss {row['test_log_loss']:.4f} | train {train_time:.1f}s | "
              f"[{done}/8 models, elapsed {fmt(el)}]", flush=True)

    res = pd.DataFrame(summary)

    # ---------------- HMM stage
    print(f"\n=== HMM / VITERBI (decoder tuned on out-of-fold train predictions; "
          f"grid of {len(list(itertools.product(*DECODER_GRID.values())))} settings per model) ===")
    hmm_rows, grid_rows = [], []
    for name in models:
        tg = time.time()
        g = tune_decoder(metas_tr, oof_seq[name])
        g.insert(0, "model", name)
        grid_rows.append(g)
        best = g.iloc[0]
        tab = transition_table(best.hop_pen, best.jump_pen, best.uturn_pen)
        clf_te = seq_metrics(metas_te, test_seq[name], tab, best.emission_weight, "clf")
        hmm_te = seq_metrics(metas_te, test_seq[name], tab, best.emission_weight, "hmm")
        clf_oof = seq_metrics(metas_tr, oof_seq[name], tab, best.emission_weight, "clf")
        hmm_rows.append(dict(
            model=name, hop_pen=best.hop_pen, jump_pen=best.jump_pen, uturn_pen=best.uturn_pen,
            emission_weight=best.emission_weight,
            oof_top1_clf=clf_oof["top1"], oof_top1_hmm=best.top1,
            test_top1_clf=clf_te["top1"], test_top1_hmm=hmm_te["top1"],
            test_top1_gain=hmm_te["top1"] - clf_te["top1"],
            test_continuity_clf=clf_te["continuity"], test_continuity_hmm=hmm_te["continuity"],
            test_route_f1_clf=clf_te["route_f1"], test_route_f1_hmm=hmm_te["route_f1"]))
        print(f"{name:26s} test Top-1 {clf_te['top1']:.4f} -> {hmm_te['top1']:.4f} (HMM) | continuity "
              f"{clf_te['continuity']:.3f} -> {hmm_te['continuity']:.3f} | route F1 {clf_te['route_f1']:.3f} -> "
              f"{hmm_te['route_f1']:.3f} | {fmt(time.time() - tg)}", flush=True)
    hmm = pd.DataFrame(hmm_rows)
    hmm.to_csv(OUTPUT_DIR / "hmm_comparison.csv", index=False)
    pd.concat(grid_rows).to_csv(OUTPUT_DIR / "decoder_grid_results.csv", index=False)

    res = res.merge(hmm[["model", "test_top1_hmm", "test_continuity_hmm", "test_route_f1_hmm"]], on="model")
    res = res.sort_values(["cv_top1_mean", "cv_logloss_mean"], ascending=[False, True])
    res.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(OUTPUT_DIR / "cv_fold_results.csv", index=False)
    json.dump(test_conf, open(OUTPUT_DIR / "test_confusion_matrices.json", "w"), indent=2)
    json.dump(dict(seed=SEED, quick_mode=QUICK_MODE, cand_radius_m=CAND_RADIUS_M, max_candidates=MAX_CANDIDATES,
                   align_radius_m=ALIGN_RADIUS_M, label_max_dist_m=LABEL_MAX_DIST_M,
                   max_points_per_trip=MAX_POINTS_PER_TRIP, cv_folds=CV_FOLDS, seq_len_train=SEQ_LEN_TRAIN,
                   seq_len_test=SEQ_LEN_TEST, max_hops=MAX_HOPS, gap_s=GAP_S, decoder_grid=DECODER_GRID,
                   features=FEATURES, models={k: dict(family=v[0], params=v[1]) for k, v in models.items()}),
              open(OUTPUT_DIR / "run_config.json", "w"), indent=2)

    show = ["model", "cv_top1_mean", "cv_top1_std", "cv_logloss_mean", "test_top1_acc", "test_top1_hmm",
            "test_f1", "test_log_loss", "train_time_s"]
    print("\n=== RANKING (by CV Top-1, tie-break CV log loss; model choice uses CV only, not the test set) ===")
    print(res[show].round(4).to_string(index=False))
    print(f"\nBest by primary metric: {res.iloc[0]['model']}")
    print(f"Total run time: {fmt(time.time() - T0)} | results saved in: {OUTPUT_DIR.resolve()}")
    return res, hmm


## Run everything

In [20]:
results, hmm_results = main()
results

Upload your Train_Test.zip ...


Saving Train&Test.zip to Train&Test.zip
Extracting Train&Test.zip ...
Found 80 train trips and 10 test trips under /content/pathsnap_work/data
  processed 10/90 trips | elapsed 5s | est. remaining 43s
  processed 20/90 trips | elapsed 9s | est. remaining 32s
  processed 30/90 trips | elapsed 12s | est. remaining 25s
  processed 40/90 trips | elapsed 16s | est. remaining 20s
  processed 50/90 trips | elapsed 19s | est. remaining 15s
  processed 60/90 trips | elapsed 23s | est. remaining 11s
  processed 70/90 trips | elapsed 26s | est. remaining 7s
  processed 80/90 trips | elapsed 30s | est. remaining 3s
  processed 90/90 trips | elapsed 35s | est. remaining 0s

=== DATASET INVENTORY ===
       trips  gps_points  median_points  median_nodes  median_arcs  unreliable_labels  max_gap_s  mean_pct_1s
split                                                                                                        
test      10       19160         1420.0        6596.5      13145.0                 2

,model,family,config,cv_top1_mean,cv_top1_std,cv_f1_mean,cv_logloss_mean,cv_logloss_std,cv_miss_rate,test_accuracy,...,test_top1_acc,test_top1_acc_given_hit,test_candidate_miss_rate,train_time_s,infer_time_s,infer_us_per_row,total_cv_time_s,test_top1_hmm,test_continuity_hmm,test_route_f1_hmm
5,6_XGBoost,Boosting,"{""n_estimators"": 400, ""learning_rate"": 0.08, ""...",0.883687,0.011777,0.876133,0.063995,0.007314,0.005777,0.978782,...,0.895132,0.895596,0.000518,17.744206,0.326375,8.414540,95.518561,0.935504,0.998903,0.958808
4,5_LightGBM,Boosting,"{""n_estimators"": 400, ""learning_rate"": 0.05, ""...",0.881992,0.013026,0.876028,0.063871,0.006990,0.005777,0.978163,...,0.894355,0.894819,0.000518,18.867463,1.069992,27.586346,131.262819,0.941985,0.999008,0.958305
2,3_RandomForest,Ensemble,"{""n_estimators"": 200, ""max_depth"": 16, ""min_sa...",0.874880,0.013269,0.870264,0.066765,0.006576,0.005777,0.977028,...,0.887882,0.888342,0.000518,269.280677,0.669853,17.270038,1352.504576,0.930800,0.999112,0.955062
6,7_MLP,Neural network,"{""hidden_layer_sizes"": [64, 32], ""alpha"": 0.00...",0.872810,0.014027,0.861923,0.071443,0.007423,0.005777,0.976255,...,0.887105,0.887565,0.000518,83.195186,0.068732,1.772044,286.717551,0.916584,0.998747,0.953320
3,4_ExtraTrees,Tree/Ensemble,"{""n_estimators"": 200, ""max_depth"": 18, ""min_sa...",0.868077,0.014464,0.859185,0.074734,0.006309,0.005777,0.975739,...,0.879855,0.880311,0.000518,52.561513,0.852832,21.987569,272.534726,0.931689,0.998956,0.949425
7,8_LinearSVM_calibrated,Kernel-based (linear),"{""C"": 0.5, ""cv"": 3}",0.853444,0.013375,0.832582,0.091339,0.005978,0.005777,0.971202,...,0.867685,0.868135,0.000518,5.449733,0.043917,1.132259,31.461687,0.925992,0.998799,0.941796
0,1_LogisticRegression,Linear/probabilistic,"{""C"": 1.0, ""max_iter"": 500}",0.852848,0.014168,0.833380,0.091417,0.006114,0.005777,0.971563,...,0.866132,0.866580,0.000518,2.100181,0.023176,0.597531,13.350648,0.927612,0.998747,0.942371
1,2_GaussianNB,Linear/probabilistic,"{""var_smoothing"": 1e-09}",0.831264,0.015231,0.613023,0.638099,0.019414,0.005777,0.880785,...,0.836095,0.836528,0.000518,0.171021,0.018503,0.477034,3.086949,0.904354,0.997389,0.931667


## Classifier-only vs classifier + HMM (test trips)

In [21]:
hmm_results[['model','test_top1_clf','test_top1_hmm','test_top1_gain','test_continuity_clf','test_continuity_hmm','test_route_f1_clf','test_route_f1_hmm']].round(4)

,model,test_top1_clf,test_top1_hmm,test_top1_gain,test_continuity_clf,test_continuity_hmm,test_route_f1_clf,test_route_f1_hmm
0,1_LogisticRegression,0.8607,0.9276,0.0669,0.9352,0.9987,0.8767,0.9424
1,2_GaussianNB,0.8253,0.9044,0.0791,0.9031,0.9974,0.8698,0.9317
2,3_RandomForest,0.8790,0.9308,0.0518,0.9380,0.9991,0.9010,0.9551
3,4_ExtraTrees,0.8690,0.9317,0.0627,0.9275,0.9990,0.8828,0.9494
4,5_LightGBM,0.8864,0.9420,0.0556,0.9433,0.9990,0.9121,0.9583
5,6_XGBoost,0.8885,0.9355,0.0470,0.9447,0.9989,0.9164,0.9588
6,7_MLP,0.8760,0.9166,0.0406,0.9355,0.9987,0.9038,0.9533
7,8_LinearSVM_calibrated,0.8613,0.9260,0.0647,0.9356,0.9988,0.8784,0.9418


## Download results (Colab)

In [22]:
import shutil
from google.colab import files
shutil.make_archive('/content/pathsnap_outputs', 'zip', str(OUTPUT_DIR))
files.download('/content/pathsnap_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>